In [134]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import json
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from '/home/sagemaker-user/analysis-tools/src/training/gbm_model_trainer.py'>

## Load data

In [3]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight functionality of plot_target_vs_predictors()
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(df["random"]  > 0.70, "V", "T")

In [6]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.698834
V    0.301166
Name: proportion, dtype: float64

In [7]:
df.groupby("split")["target"].mean()

split
T    0.116981
V    0.116995
Name: target, dtype: float64

In [8]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])

## Use ModelTrainer class for training
- Test hyperparameter tuning

In [135]:
# Define your config
config = GBMModelTrainerConfig(
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },
    output_log=True,
    output_report=True,
    output_email=True,
    log_file="outputs/training.log",
    report_output_path="outputs/model_analysis.html",
    report_params={
    },
    email="asacco@plymouthrock.com",
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Choose your output path
config_output_path = Path("model_config.json")

# Write config to JSON file
with config_output_path.open("w") as f:
    json.dump(asdict(config), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")

✅ Config saved to /home/sagemaker-user/analysis-tools/notebooks/model_config.json


In [142]:
reload(training.gbm_model_trainer)
reload(analysis.report)

<module 'analysis.report' from '/home/sagemaker-user/analysis-tools/src/analysis/report.py'>

In [143]:
from training.gbm_model_trainer import GBMModelTrainer
from xgboost import XGBClassifier

mt_xgboost = GBMModelTrainer(
    model_class=XGBClassifier,
    config_path="model_config.json",
    train_df=train,
    valid_df=test
)

In [138]:
mt_xgboost.model_class

xgboost.sklearn.XGBClassifier

In [139]:
mt_xgboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', log_file='outputs/training.log', report_output_path='outputs/model_analysis.html', email='asacco@plymouthrock.com', hyperparameters={'objective': 'binary:logistic', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}, output_log=True, output_email=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type': 'normalized

In [144]:
mt_xgboost.train()

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/xgboost/core.py:160: UserWarning: [15:51:18] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "early_stopping" } are not used.

  warnings.warn(smsg, UserWarning)
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_

✅ Analysis report generated at outputs/model_analysis.html


In [145]:
mt_xgboost.model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5, device=None, early_stopping=25,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric=None, feature_types=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

## Test with CAT Boost

In [ ]:
# Define your config
config = GBMModelTrainerConfig(
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },
    output_log=True,
    output_report=True,
    output_email=True,
    log_file="outputs/training.log",
    report_output_path="outputs/model_analysis.html",
    report_params={
    },
    email="asacco@plymouthrock.com",
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Choose your output path
config_output_path = Path("model_config.json")

# Write config to JSON file
with config_output_path.open("w") as f:
    json.dump(asdict(config), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")